<a href="https://colab.research.google.com/github/BishmoyPaul/LLM_Finetuning_Tutorials/blob/colabnotebookfix/Llama_3_2_Finetuning_using_ORPO_and_Customized_LoRA_layers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook provides a short tutorial on fine-tuning LLaMA 3.2 1B models using ORPO with custom LoRA layers (using NDLinear layers), and measuring the resulting performance changes.

**And yes, you can run all of it in the Free Tier of Google Colab (Tesla T4)**

We will walk through the following steps:


1.   **Baseline Evaluation**: Load the LLaMA 3.2 1B model and evaluate its performance using [LM Evaluation Harness](https://github.com/EleutherAI/lm-evaluation-harness) and [tinybenchmarks](https://github.com/felipemaiapolo/tinyBenchmarks)
2.   **Custom LoRA Layers**: Implement custom LoRA layers using [NDLinear](https://arxiv.org/abs/2503.17353) modules for fine-tuning.
3. **Model Training**: Fine-tune the model with ORPO ([Odds Ratio Preference Optimization](https://arxiv.org/abs/2403.07691)) on a small subset of the `mlabonne/orpo-dpo-mix-40k` dataset.
4. **Post-Tuning Evaluation**: Re-evaluate the model to measure the impact of fine-tuning using the same benchmarks as the baseline.

## Installing Libraries

In [ ]:
# installing model libraries and fine-tuning
!pip install -qqq -U transformers datasets accelerate peft trl bitsandbytes --progress-bar off
# installing ndlinear - which would be the basis for our custom lora layer
!pip install -qqq ndlinear --progress-bar off
# installing evaluation tools
!pip install -qqq lm_eval git+https://github.com/felipemaiapolo/tinyBenchmarks

## Importing Libraries

In [ ]:
import os
import gc

import torch
import torch.nn as nn

# for progress bar
from tqdm.notebook import tqdm

# for downloading our dataset
from datasets import load_dataset

# for authenticating
from google.colab import userdata

# for downloading model and fine-tuning
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
)
from trl import ORPOConfig, ORPOTrainer, setup_chat_format

# for custom model layer
from ndlinear import NdLinear

# for evaluation
from lm_eval.models.huggingface import HFLM
from lm_eval import evaluator
from lm_eval.tasks import get_task_dict

## Setting up initial parameters

In [ ]:
# Model
base_model = "meta-llama/Llama-3.2-1B"
new_model = "Llama-3.2-1B-NDLinear-LoRA-ORPO"

# for tesla T4 GPU
torch_dtype = torch.float16
attn_implementation = "eager"

# for gated repos like llama 3.2 1B, we need to authenticate using HF token
from google.colab import userdata
hf_token = userdata.get('HF_MODEL')

# need login for downloading llama models
from huggingface_hub import login
login(token=hf_token)

## Loading the base model

In [ ]:
# QLoRA config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation=attn_implementation,
)

# Time to load our model and tokenizer
model, tokenizer = setup_chat_format(model, tokenizer)

## Evaluating the base model

In [ ]:
# generate a model object of lm-harness evaluation
model_obj = HFLM(
    pretrained=model.eval(),
    tokenizer=tokenizer,
    batch_size=1, # can be adjusted based on memory constraints
    max_length=2048,
)

# we pick tiny benchmarks, bigger benchmarks would take more time
task = "tinyTruthfulQA"

results = evaluator.simple_evaluate(
    model=model_obj,
    tasks=[task],
    num_fewshot=0,
    )

print("\nBaseline results : ",results['results'][task]['acc,none'])

In [ ]:
# [optional] clean up
del model_obj
gc.collect()
gc.collect()
torch.cuda.empty_cache()

## Building our custom LoRA layer

Here we build our custom LoRA layer. The some of the customizations here are NDLinear specific (like removing `relu` and setting `bias` to 0), but you can build your own layers based on your own specificiations. Go Wild!

In [ ]:
class CustomLoraReplacement(nn.Module):
    def __init__(self, in_features, out_features, lora_type, lora_alpha=32, r=16, lora_dropout = 0.05):
        super().__init__()

        self.lora_type = lora_type # 'A' or 'B'
        # for lora_type 'A', we would set in_features = r (rank),
        # for lora_type 'B', we would set out_features = r (rank)

        # now we initialize our custom layer
        self.layer = NdLinear(input_dims=(in_features,), hidden_size=(out_features,))

        # incase the NDlinear layer still has relu at start, we want to remove it
        if hasattr(self.layer, 'relu'):
          delattr(self.layer, 'relu')


        # since we are only intializing a single layer for this, align_layers would have a length of 1
        # we want to disable its bias
        if self.layer.align_layers[0].bias is not None:
          nn.init.zeros_(self.layer.align_layers[0].bias)
          self.layer.align_layers[0].bias.requires_grad = False


        assert len(self.layer.align_layers) == 1, "Need to adjust PEFT methods for more layers "


        self.scaling = 1.0

        if lora_type == 'A':
            # For lora_A (r, input_features) - transposed for nn.Linear
            # NDLinear contains multiple smallaer linear layers, so we initialize them
            # The weights need to be uniform
            nn.init.uniform_(self.layer.align_layers[0].weight, -0.1, 0.1)

            # we also need to incorporate dropout
            self.dropout = nn.Dropout(p=lora_dropout)


        elif lora_type == 'B':
            # For lora_B (output_features, r)
            nn.init.zeros_(self.layer.align_layers[0].weight)

            # only apply scaling to B, as it would be A * B * scale
            # applying it to A would make it A * scale * B * scale
            self.scaling = lora_alpha / r

        else:
          raise ValueError("Not Implemented")

        self.weight = self.layer.align_layers[0].weight


    def forward(self, x):
        if self.lora_type == 'A':
            return self.dropout(self.layer(x))

        return self.layer(x) * self.scaling


To use our custom layers, we can initialize default loras (lora A and lora B) using the PEFT, and replace those default layers with layers we define. The advantage of this is that we don't need to manually track which layers we want to replace, PEFT does that for us, we need to only change those layers to the layers we define.

In [ ]:
def replace_lora_layers(model, lora_alpha = 32, r = 16, lora_dropout = 0.05, verbose = 0):
    # Track LoRA modules and their paths for replacement
    modules_to_replace = []

    # Find all LoRA modules
    for name, module in model.named_modules():
        # Identify lora_A and lora_B modules
        if name.endswith('lora_A') or name.endswith('lora_A.default'):

            device = next(module.parameters()).device
            # Get the actual shape of the weight, we need this for initializing our layers
            weight_shape = None
            for param_name, param in module.named_parameters():
                if param_name == 'weight':
                    weight_shape = param.shape
                    break

            if weight_shape is not None:
                modules_to_replace.append({
                    'full_name': name,
                    'type': 'A',
                    'shape': weight_shape,  # (r, in_features)
                    'in_features': weight_shape[1],
                    'out_features': weight_shape[0],  # r.
                    'device': device
                })

        elif name.endswith('lora_B') or name.endswith('lora_B.default'):

            device = next(module.parameters()).device
            # Get the actual shape of the weight
            weight_shape = None
            for param_name, param in module.named_parameters():
                if param_name == 'weight':
                    weight_shape = param.shape
                    break

            if weight_shape is not None:
                modules_to_replace.append({
                    'full_name': name,
                    'type': 'B',
                    'shape': weight_shape,  # (out_features, r)
                    'in_features': weight_shape[1],  # r
                    'out_features': weight_shape[0],  # out_features
                    'device': device
                })

    # Now replace each module
    for info in modules_to_replace:
        full_name = info['full_name']
        lora_type = info['type']
        in_features = info['in_features']
        out_features = info['out_features']
        device = info['device']

        # Split the name to navigate to the parent
        path_parts = full_name.split('.')

        # Navigate to the parent module
        current_module = model
        for i, part in enumerate(path_parts):
            if i < len(path_parts) - 1:
                # Navigate to the next level
                if hasattr(current_module, part):
                    current_module = getattr(current_module, part)
                else:
                    # Handle dictionary-like modules
                    current_module = current_module[part]
            else:
                # This is the last part (the module to replace) - gotcha!
                target_name = part

        # Initialize our lora
        custom_layer = CustomLoraReplacement(
            in_features=in_features,
            out_features=out_features,
            lora_type=lora_type,
            lora_alpha=lora_alpha,
            r=r,
            lora_dropout = lora_dropout
        ).to(device)

        # Time to replace that layer with our LoRA
        if hasattr(current_module, target_name):
            setattr(current_module, target_name, custom_layer)
        else:
            # For dictionary-like modules
            current_module[target_name] = custom_layer

        if verbose:
          print(f"Replaced {full_name} with CustomLoraReplacement (type={lora_type})")

    return model

In [ ]:
# LoRA config
lora_alpha = 32
r = 16
lora_dropout = 0.05

peft_config = LoraConfig(
    r=r,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=['up_proj', 'down_proj', 'gate_proj', 'k_proj', 'q_proj', 'v_proj', 'o_proj']
)

# First, apply standard PEFT to get the structure
peft_model = get_peft_model(model, peft_config)

# Then replace the LoRA layers with custom ones
custom_model = replace_lora_layers(peft_model, lora_alpha=lora_alpha, r=r, lora_dropout = lora_dropout)

# Finally, we need to prepare our model for kbit training
custom_model = prepare_model_for_kbit_training(custom_model)

In [ ]:
# we need to make the replaced lora layers trainable
for name, param in custom_model.named_parameters():
    if 'lora' in name:
      if "default.weight" in name:
        param.requires_grad = True

## Loading our training dataset

In [ ]:
dataset_name = "mlabonne/orpo-dpo-mix-40k"
dataset = load_dataset(dataset_name, split="all")
dataset = dataset.shuffle(seed=42).select(range(1000)) # Only use 1000 samples for quick demo

In [ ]:
def format_chat_template(row):
    row["chosen"] = tokenizer.apply_chat_template(row["chosen"], tokenize=False)
    row["rejected"] = tokenizer.apply_chat_template(row["rejected"], tokenize=False)
    return row

dataset = dataset.map(
    format_chat_template,
    num_proc= os.cpu_count(),
)
dataset = dataset.train_test_split(test_size=0.01)

## Training!

In [ ]:
orpo_args = ORPOConfig(
    learning_rate=8e-6,
    lr_scheduler_type="linear",
    max_length=1024,
    max_prompt_length=512,
    beta=0.1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_8bit",
    max_steps=20,  # to keep our demo short, we are training for 20 steps only # takes around 6 minutes on Tesla T4
    #num_train_epochs=1, # to train for a full epoch, uncomment this and comment out max_steps
    eval_steps=0.2,
    logging_steps=1,
    warmup_steps=10,
    report_to="none",
    output_dir="./results/",
)

trainer = ORPOTrainer(
    model=custom_model,
    args=orpo_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

## Evaluating our model performance

In [ ]:
# generate a model object again
model_obj_new = HFLM(
    pretrained=custom_model.eval(),
    tokenizer=tokenizer,
    batch_size=1,
    max_length=2048
)


results_new = evaluator.simple_evaluate(
    model=model_obj_new,
    tasks=[task],
    num_fewshot=0,
    )

print("\nFine-tune results : ",results_new['results'][task]['acc,none'])

## References

1. [Fine-tune Llama 3 with ORPO](https://huggingface.co/blog/mlabonne/orpo-llama-3) - The tutorial and dataset (`mlabonne/orpo-dpo-mix-40k`) that served as the foundation for this implementation. Thanks to [@mlabonne](https://huggingface.co/mlabonne) for his amazing resources!
2. [NDLinear](https://github.com/ensemble-core/NdLinear) - The layer used for our custom LoRA
3. 🤗 Hugging Face Transformers, PEFT and TRL Libraries - Leveraged for model loading, fine-tuning pipelines, and training infrastructure
4. [LM Evaluation Harness](https://github.com/EleutherAI/lm-evaluation-harness) and [tinybenchmarks](https://github.com/felipemaiapolo/tinyBenchmarks) - Utilized for quantitative evaluation of baseline and fine-tuned model performance

-